<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/15_ctd_tradeoff_mitigation_dpo_mixture_confidence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 15 — CTD Selection–Sufficiency Trade-off Mitigation

Additional experiment for the reliable biomedical evidence reasoning paper.

We compare three interventions on the primary Qwen2.5-0.5B task:

1. **Mixture SFT** — one model sees distractor-positive and explicit no-path examples.
2. **DPO** — start from a distractor-robust SFT adapter and optimize preferences for supported answers vs. unsupported/distractor answers.
3. **Confidence abstention** — no additional preference training; calibrate a confidence threshold on the robust SFT model and abstain at low confidence.

The notebook runs 3 entity-disjoint splits × 3 seeds and evaluates Clean / Distractor-5 / Hard no-path / Lexical no-path / Counterfactual.

## Persistence / resume behavior
- Google Drive is mounted **before training**.
- `results/15/15_results.csv` and `15_summary.csv` are atomically mirrored to Drive after **every completed method**.
- Robust LoRA adapters are persisted to Drive because they are reused by DPO and confidence calibration.
- On restart, completed `(split, seed, method)` blocks are skipped automatically.
- If a runtime dies during one training block, only that unfinished block needs to be repeated.


In [1]:
!pip -q install -U transformers datasets trl peft accelerate bitsandbytes sentencepiece requests

import os, re, gc, json, gzip, random, inspect, shutil, tempfile, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch, requests
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    set_seed, TrainingArguments
)
from peft import (
    LoraConfig, PeftModel, prepare_model_for_kbit_training
)
from trl import SFTConfig, SFTTrainer, DPOConfig, DPOTrainer

# -----------------------------
# Experiment configuration
# -----------------------------
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
SEEDS = [1, 2, 3]
SPLITS = ["ChemicalID", "GeneID", "DiseaseID"]
N_TRAIN = 1500
N_CAL = 80
N_EVAL = 100
MAX_STEPS_SFT = 80
MAX_STEPS_DPO = 60
MIX_NO_PATH_FRAC = 0.25
MIX_LEXICAL_FRAC = 0.50
DPO_NO_PATH_FRAC = 0.50
GEN_MAX_NEW_TOKENS = 48

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
DATA_DIR = ROOT / "ctd_data"
LOCAL_RESULT_DIR = ROOT / "results" / "15"
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Mount Drive before any long work.
DRIVE_RESULT_DIR = None
if Path("/content").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_RESULT_DIR = Path("/content/drive/MyDrive/llm-tuning-playground/results/15")
        DRIVE_RESULT_DIR.mkdir(parents=True, exist_ok=True)
        print("Persistent results:", DRIVE_RESULT_DIR)
    except Exception as e:
        print("Drive mount failed; local-only mode:", repr(e))

RESULT_CSV = LOCAL_RESULT_DIR / "15_results.csv"
SUMMARY_CSV = LOCAL_RESULT_DIR / "15_summary.csv"
CONFIG_JSON = LOCAL_RESULT_DIR / "15_config.json"
PERSIST_RESULT_CSV = (DRIVE_RESULT_DIR / "15_results.csv") if DRIVE_RESULT_DIR else None
PERSIST_SUMMARY_CSV = (DRIVE_RESULT_DIR / "15_summary.csv") if DRIVE_RESULT_DIR else None
PERSIST_CONFIG_JSON = (DRIVE_RESULT_DIR / "15_config.json") if DRIVE_RESULT_DIR else None
ADAPTER_ROOT = (DRIVE_RESULT_DIR / "adapters") if DRIVE_RESULT_DIR else (LOCAL_RESULT_DIR / "adapters")
ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)

config = {
    "experiment": 15,
    "model": MODEL_NAME,
    "seeds": SEEDS,
    "splits": SPLITS,
    "n_train": N_TRAIN,
    "n_cal": N_CAL,
    "n_eval": N_EVAL,
    "max_steps_sft": MAX_STEPS_SFT,
    "max_steps_dpo": MAX_STEPS_DPO,
    "mixture_no_path_frac": MIX_NO_PATH_FRAC,
    "mixture_lexical_frac": MIX_LEXICAL_FRAC,
    "dpo_no_path_frac": DPO_NO_PATH_FRAC,
    "methods": ["robust_reference", "mixture_sft", "dpo", "confidence_abstention"],
}
CONFIG_JSON.write_text(json.dumps(config, indent=2))
if PERSIST_CONFIG_JSON:
    PERSIST_CONFIG_JSON.write_text(json.dumps(config, indent=2))

print("Model:", MODEL_NAME)
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Use a Colab GPU runtime for this experiment."


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 136.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 50.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Mounted at /content/drive
Persistent results: /content/drive/MyDrive/llm-tuning-playground/results/15
Model: Qwen/Qwen2.5-0.5B-Instruct
CUDA: NVIDIA L4


## CTD acquisition and parsing

This reuses the robust CTD parser from Experiment 14: CTD bulk files store the actual header in a commented line, so the header is recovered explicitly before reading the data.


In [2]:
CHEM_NAME = "CTD_chem_gene_ixns.tsv.gz"
GD_NAMES = ["CTD_curated_genes_diseases.tsv.gz", "CTD_genes_diseases.tsv.gz"]

def valid_gzip(path, min_bytes=10000):
    path = Path(path)
    if not path.exists() or path.stat().st_size < min_bytes:
        return False
    try:
        with open(path, "rb") as f:
            if f.read(2) != b"\x1f\x8b":
                return False
        with gzip.open(path, "rb") as f:
            f.read(256)
        return True
    except Exception:
        return False

def find_local(name):
    candidates = [
        Path.cwd()/name, ROOT/name, DATA_DIR/name,
        Path("/content/drive/MyDrive")/name,
        Path("/content/drive/MyDrive/ctd")/name,
        Path("/content/drive/MyDrive/data")/name,
    ]
    for p in candidates:
        if valid_gzip(p):
            print("Found:", p)
            return p
    return None

def download_ctd(name):
    dest = DATA_DIR / name
    urls = [
        f"https://ctdbase.org/reports/{name}",
        f"https://ctdbase.org/downloads/{name}",
        f"http://ctdbase.org/reports/{name}",
    ]
    for url in urls:
        try:
            print("Trying:", url)
            with requests.get(
                url, stream=True, timeout=(20, 300), allow_redirects=True,
                headers={"User-Agent": "Mozilla/5.0"}
            ) as r:
                r.raise_for_status()
                with open(dest, "wb") as f:
                    for chunk in r.iter_content(1024*1024):
                        if chunk:
                            f.write(chunk)
            if valid_gzip(dest):
                print("Downloaded:", dest)
                return dest
        except Exception as e:
            print(" failed:", type(e).__name__, str(e)[:120])
        dest.unlink(missing_ok=True)
    return None

def ensure_ctd(names):
    if isinstance(names, str):
        names = [names]
    for n in names:
        p = find_local(n)
        if p:
            return p
    for n in names:
        p = download_ctd(n)
        if p:
            return p
    try:
        from google.colab import files
        print("Automatic download failed. Upload one of:", names)
        up = files.upload()
        for n in names:
            if n in up:
                p = DATA_DIR / n
                p.write_bytes(up[n])
                if valid_gzip(p):
                    return p
    except Exception:
        pass
    raise FileNotFoundError("Could not obtain CTD data: " + ", ".join(names))

def read_ctd(path, expected_any):
    header = None
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            if not line.startswith("#"):
                break
            s = line.lstrip("#").strip()
            if "\t" in s:
                cols = [x.strip() for x in s.split("\t")]
                if any(x in cols for x in expected_any):
                    header = cols
    if header is None:
        raise ValueError(f"Could not recover CTD header from {path}")
    return pd.read_csv(
        path, sep="\t", comment="#", compression="gzip",
        dtype=str, low_memory=False, header=None, names=header
    )

def pick(df, names):
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(f"None of {names} found. Columns={list(df.columns)[:30]}")

CHEM_GENE = ensure_ctd(CHEM_NAME)
GENE_DISEASE = ensure_ctd(GD_NAMES)
cg = read_ctd(CHEM_GENE, ["ChemicalName","ChemicalID","GeneSymbol","GeneID"])
gd = read_ctd(GENE_DISEASE, ["GeneSymbol","GeneID","DiseaseName","DiseaseID"])

c_name = pick(cg, ["ChemicalName"]); c_id = pick(cg, ["ChemicalID"])
g_sym1 = pick(cg, ["GeneSymbol"]); g_id1 = pick(cg, ["GeneID"])
g_sym2 = pick(gd, ["GeneSymbol"]); g_id2 = pick(gd, ["GeneID"])
d_name = pick(gd, ["DiseaseName"]); d_id = pick(gd, ["DiseaseID"])

cg2 = cg[[c_name,c_id,g_sym1,g_id1]].dropna().drop_duplicates()
gd2 = gd[[g_sym2,g_id2,d_name,d_id]].dropna().drop_duplicates()
cg2.columns = ["ChemicalName","ChemicalID","GeneSymbol","GeneID"]
gd2.columns = ["GeneSymbol","GeneID","DiseaseName","DiseaseID"]

paths = cg2.merge(gd2, on=["GeneSymbol","GeneID"], how="inner").drop_duplicates()
paths = paths[
    (paths.ChemicalName.str.len() < 100) &
    (paths.DiseaseName.str.len() < 120)
] .reset_index(drop=True)
edge_pool = gd2[["GeneSymbol","DiseaseName"]].drop_duplicates().reset_index(drop=True)

assert len(paths) > 5000, f"Too few joined paths: {len(paths)}"
print("Two-hop paths:", len(paths))
display(paths.head())


Trying: https://ctdbase.org/reports/CTD_chem_gene_ixns.tsv.gz
Downloaded: /content/ctd_data/CTD_chem_gene_ixns.tsv.gz
Trying: https://ctdbase.org/reports/CTD_curated_genes_diseases.tsv.gz
Downloaded: /content/ctd_data/CTD_curated_genes_diseases.tsv.gz
Two-hop paths: 9707313


,ChemicalName,ChemicalID,GeneSymbol,GeneID,DiseaseName,DiseaseID
0,10074-G5,C534883,AR,367,Alopecia,MESH:D000505
1,10074-G5,C534883,AR,367,Androgen-Insensitivity Syndrome,MESH:D013734
2,10074-G5,C534883,AR,367,Astrocytoma,MESH:D001254
3,10074-G5,C534883,AR,367,Autistic Disorder,MESH:D001321
4,10074-G5,C534883,AR,367,Breast Neoplasms,MESH:D001943


In [3]:
# -----------------------------
# Controlled task construction
# -----------------------------
def render_prompt(row, edges):
    lines = [f"- {g} -> {d}" for g, d in edges]
    return (
        "Use only the supplied evidence. Determine the disease supported by the path "
        "from the queried chemical through the queried gene. If no supplied gene-disease "
        "relation supports the queried gene, answer exactly: No supported path.\n\n"
        f"Chemical: {row.ChemicalName}\n"
        f"Gene: {row.GeneSymbol}\n"
        "Evidence:\n" + "\n".join(lines)
    )

def answer_text(row):
    return (
        f"Disease: {row.DiseaseName}. Reasoning: "
        f"{row.ChemicalName} -> {row.GeneSymbol} -> {row.DiseaseName}."
    )

def positive_edges(row, k, rng):
    edges = [(str(row.GeneSymbol), str(row.DiseaseName))]
    pool = edge_pool[
        (edge_pool.GeneSymbol != row.GeneSymbol) &
        (edge_pool.DiseaseName != row.DiseaseName)
    ]
    if k:
        sub = pool.sample(n=k, random_state=rng.randint(0, 2**31-1))
        edges += [(str(g),str(d)) for g,d in sub.itertuples(index=False,name=None)]
    rng.shuffle(edges)
    return edges

def no_path_edges(row, k, rng, lexical=False):
    pool = edge_pool[
        (edge_pool.GeneSymbol != row.GeneSymbol) &
        (edge_pool.DiseaseName != row.DiseaseName)
    ].copy()
    edges = []
    if lexical:
        sym = str(row.GeneSymbol)
        prefix = sym[:max(1, min(2, len(sym)))]
        near = pool[pool.GeneSymbol.astype(str).str.startswith(prefix)]
        if len(near):
            x = near.sample(1, random_state=rng.randint(0,2**31-1)).iloc[0]
            edges.append((str(x.GeneSymbol), str(x.DiseaseName)))
            pool = pool[pool.GeneSymbol != x.GeneSymbol]
    need = k - len(edges)
    if need > 0:
        sub = pool.sample(need, random_state=rng.randint(0,2**31-1))
        edges += [(str(g),str(d)) for g,d in sub.itertuples(index=False,name=None)]
    rng.shuffle(edges)
    return edges

def counterfactual_edges(row, rng):
    c = edge_pool[
        (edge_pool.GeneSymbol == row.GeneSymbol) &
        (edge_pool.DiseaseName != row.DiseaseName)
    ]
    if len(c):
        cf = str(c.sample(1, random_state=rng.randint(0,2**31-1)).iloc[0].DiseaseName)
    else:
        cf = str(
            edge_pool[edge_pool.DiseaseName != row.DiseaseName]
            .sample(1, random_state=rng.randint(0,2**31-1)).iloc[0].DiseaseName
        )
    return [(str(row.GeneSymbol), cf)], cf

def make_split(df, col, seed):
    r = np.random.default_rng(seed)
    ents = df[col].dropna().unique().copy()
    r.shuffle(ents)
    cut = max(1, int(0.8 * len(ents)))
    tr_e, te_e = set(ents[:cut]), set(ents[cut:])
    trp = df[df[col].isin(tr_e)]
    tep = df[df[col].isin(te_e)].drop_duplicates(["ChemicalID","GeneID","DiseaseID"])
    assert len(trp) >= N_TRAIN
    assert len(tep) >= N_CAL + N_EVAL
    tr = trp.sample(N_TRAIN, random_state=seed).reset_index(drop=True)
    held = tep.sample(N_CAL + N_EVAL, random_state=1000+seed).reset_index(drop=True)
    cal = held.iloc[:N_CAL].reset_index(drop=True)
    te = held.iloc[N_CAL:].reset_index(drop=True)
    assert set(tr[col]).isdisjoint(set(pd.concat([cal,te])[col]))
    return tr, cal, te

def item(row, edges, target, typ):
    return {
        "target_gene": str(row.GeneSymbol),
        "target_disease": None if target is None else str(target),
        "evidence_edges": [(str(g),str(d)) for g,d in edges],
        "prompt": render_prompt(row, edges),
        "answer_type": typ,
    }

def make_eval_sets(df, seed):
    rng = random.Random(20000 + seed)
    out = {k: [] for k in [
        "clean","distractor_5","hard_no_path","lexical_no_path","counterfactual"
    ]}
    for _, row in df.iterrows():
        out["clean"].append(item(row, positive_edges(row,0,rng), row.DiseaseName, "positive"))
        out["distractor_5"].append(item(row, positive_edges(row,5,rng), row.DiseaseName, "positive"))
        out["hard_no_path"].append(item(row, no_path_edges(row,5,rng,False), None, "no_path"))
        out["lexical_no_path"].append(item(row, no_path_edges(row,5,rng,True), None, "no_path"))
        e, cf = counterfactual_edges(row, rng)
        out["counterfactual"].append(item(row, e, cf, "positive"))
    return out

def make_robust_sft_dataset(df, seed):
    rng = random.Random(seed)
    rec = []
    for _, row in df.iterrows():
        edges = positive_edges(row, rng.choice([1,3,5,10]), rng)
        rec.append({"text": render_prompt(row,edges) + "\nAnswer: " + answer_text(row)})
    return Dataset.from_list(rec)

def make_mixture_sft_dataset(df, seed):
    rng = random.Random(seed)
    rec = []
    for _, row in df.iterrows():
        if rng.random() < MIX_NO_PATH_FRAC:
            lexical = rng.random() < MIX_LEXICAL_FRAC
            edges = no_path_edges(row, rng.choice([3,5,10]), rng, lexical)
            ans = "No supported path."
        else:
            edges = positive_edges(row, rng.choice([1,3,5,10]), rng)
            ans = answer_text(row)
        rec.append({"text": render_prompt(row,edges) + "\nAnswer: " + ans})
    return Dataset.from_list(rec)

def make_dpo_dataset(df, seed):
    rng = random.Random(30000 + seed)
    rec = []
    for _, row in df.iterrows():
        if rng.random() < DPO_NO_PATH_FRAC:
            edges = no_path_edges(row, rng.choice([3,5,10]), rng, rng.random() < 0.5)
            rejected_disease = edges[0][1]
            chosen = "No supported path."
            rejected = f"Disease: {rejected_disease}."
        else:
            edges = positive_edges(row, rng.choice([3,5,10]), rng)
            distractors = [d for g,d in edges if g != str(row.GeneSymbol)]
            rejected_disease = distractors[0] if distractors else "Unknown disease"
            chosen = answer_text(row)
            rejected = f"Disease: {rejected_disease}."
        rec.append({
            "prompt": render_prompt(row,edges) + "\nAnswer:",
            "chosen": chosen,
            "rejected": rejected,
        })
    return Dataset.from_list(rec)


## Model, training, scoring, and persistence

The trainer wrappers filter keyword arguments against the installed TRL signatures. This avoids the version-specific `SFTConfig`/`DPOConfig` API failures encountered in earlier Colab runs.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = "left"

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

def load_base(training=False):
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb, device_map="auto"
    )
    m.config.use_cache = not training
    if training:
        m = prepare_model_for_kbit_training(m)
    return m

def load_adapter(adapter_dir, training=False):
    base = load_base(training=False)
    m = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=training)
    if training:
        m.config.use_cache = False
    return m

def only_supported_kwargs(cls, kwargs):
    sig = inspect.signature(cls.__init__)
    if any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values()):
        return kwargs
    return {k:v for k,v in kwargs.items() if k in sig.parameters}

def train_sft(ds, outdir, seed, save_adapter=None):
    set_seed(seed)
    m = load_base(training=True)
    cfg_kwargs = dict(
        output_dir=str(outdir),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        max_steps=MAX_STEPS_SFT,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        bf16=(compute_dtype == torch.bfloat16),
        fp16=(compute_dtype == torch.float16),
        gradient_checkpointing=True,
        packing=False,
        dataset_text_field="text",
        max_length=512,
        max_seq_length=512,
        warmup_steps=max(1, int(0.05*MAX_STEPS_SFT)),
    )
    args = SFTConfig(**only_supported_kwargs(SFTConfig, cfg_kwargs))
    trainer_kwargs = dict(
        model=m, args=args, train_dataset=ds,
        peft_config=lora, processing_class=tokenizer, tokenizer=tokenizer,
    )
    trainer = SFTTrainer(**only_supported_kwargs(SFTTrainer, trainer_kwargs))
    trainer.train()
    trained = trainer.model
    if save_adapter:
        save_adapter = Path(save_adapter)
        save_adapter.mkdir(parents=True, exist_ok=True)
        trained.save_pretrained(save_adapter)
        tokenizer.save_pretrained(save_adapter)
        print("Saved adapter:", save_adapter)
    return trained

def train_dpo(dpo_ds, robust_adapter, outdir, seed):
    set_seed(seed)
    m = load_adapter(robust_adapter, training=True)
    cfg_kwargs = dict(
        output_dir=str(outdir),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=8e-6,
        max_steps=MAX_STEPS_DPO,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        bf16=(compute_dtype == torch.bfloat16),
        fp16=(compute_dtype == torch.float16),
        gradient_checkpointing=True,
        beta=0.1,
        max_length=512,
        max_prompt_length=384,
        warmup_steps=max(1, int(0.05*MAX_STEPS_DPO)),
    )
    args = DPOConfig(**only_supported_kwargs(DPOConfig, cfg_kwargs))
    trainer_kwargs = dict(
        model=m,
        ref_model=None,  # PEFT DPO: TRL can use the initial adapter policy as reference.
        args=args,
        train_dataset=dpo_ds,
        processing_class=tokenizer,
        tokenizer=tokenizer,
    )
    trainer = DPOTrainer(**only_supported_kwargs(DPOTrainer, trainer_kwargs))
    trainer.train()
    return trainer.model

def norm(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())

def score_one(x, pred):
    p = norm(pred)
    if x["answer_type"] == "no_path":
        return "no supported path" in p
    return norm(x["target_disease"]) in p and "no supported path" not in p

@torch.no_grad()
def generate_predictions(model, items, return_conf=False):
    model.eval()
    preds, confs = [], []
    for x in items:
        messages = [{"role":"user","content":x["prompt"]}]
        txt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inp = tokenizer(txt, return_tensors="pt").to(model.device)
        out = model.generate(
            **inp,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=False,
            return_dict_in_generate=True,
            output_scores=return_conf,
            pad_token_id=tokenizer.eos_token_id,
        )
        gen = out.sequences[:, inp["input_ids"].shape[1]:]
        pred = tokenizer.decode(gen[0], skip_special_tokens=True)
        preds.append(pred)
        if return_conf:
            # Mean maximum next-token probability along the generated answer.
            tok_conf = []
            for scores_t, token_t in zip(out.scores, gen[0]):
                prob = torch.softmax(scores_t[0].float(), dim=-1)[int(token_t)].item()
                tok_conf.append(prob)
            confs.append(float(np.mean(tok_conf)) if tok_conf else 0.0)
    return (preds, confs) if return_conf else preds

def evaluate_model(model, eval_sets):
    scores = {}
    for name, items in eval_sets.items():
        preds = generate_predictions(model, items)
        scores[name] = float(np.mean([score_one(x,p) for x,p in zip(items,preds)]))
        print(name, scores[name])
    return scores

def calibrate_confidence_threshold(model, cal_sets):
    # Calibrate on positive distractor and hard no-path only.
    pos = cal_sets["distractor_5"]
    neg = cal_sets["hard_no_path"]
    pp, pc = generate_predictions(model, pos, return_conf=True)
    npred, nc = generate_predictions(model, neg, return_conf=True)
    candidates = np.unique(np.quantile(np.array(pc+nc), np.linspace(0,1,41)))
    best = None
    for t in candidates:
        # Below threshold => abstain; otherwise keep generated answer.
        pos_pred = ["No supported path." if c < t else p for p,c in zip(pp,pc)]
        neg_pred = ["No supported path." if c < t else p for p,c in zip(npred,nc)]
        pos_acc = np.mean([score_one(x,p) for x,p in zip(pos,pos_pred)])
        neg_acc = np.mean([score_one(x,p) for x,p in zip(neg,neg_pred)])
        objective = 0.5*(pos_acc + neg_acc)
        row = (objective, float(t), float(pos_acc), float(neg_acc))
        if best is None or row > best:
            best = row
    print("Confidence calibration:", {
        "objective":best[0],"threshold":best[1],
        "cal_d5":best[2],"cal_hard_np":best[3]
    })
    return best[1]

def evaluate_confidence_abstention(model, eval_sets, threshold):
    scores = {}
    for name, items in eval_sets.items():
        preds, confs = generate_predictions(model, items, return_conf=True)
        abstained = ["No supported path." if c < threshold else p for p,c in zip(preds,confs)]
        scores[name] = float(np.mean([score_one(x,p) for x,p in zip(items,abstained)]))
        print("confidence", name, scores[name])
    return scores

def cleanup(*models):
    for m in models:
        try:
            del m
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()

def atomic_write_df(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def load_existing_rows():
    for p in [PERSIST_RESULT_CSV, RESULT_CSV]:
        if p and Path(p).exists():
            d = pd.read_csv(p)
            print("Resuming from:", p, "rows=", len(d))
            return d.to_dict("records")
    return []

rows = load_existing_rows()

def method_complete(rows, split, seed, method):
    keys = {
        r["eval_type"] for r in rows
        if r["split"] == split and int(r["seed"]) == int(seed) and r["method"] == method
    }
    return keys >= {"clean","distractor_5","hard_no_path","lexical_no_path","counterfactual"}

def upsert_method(rows, split, seed, method, scores, extra=None):
    extra = extra or {}
    # Remove any partial stale rows for this method.
    rows[:] = [
        r for r in rows
        if not (r["split"] == split and int(r["seed"]) == int(seed) and r["method"] == method)
    ]
    for eval_type, acc in scores.items():
        rows.append({
            "split": split, "seed": int(seed), "method": method,
            "eval_type": eval_type, "accuracy": float(acc), **extra
        })
    df = pd.DataFrame(rows)
    atomic_write_df(df, RESULT_CSV)
    if PERSIST_RESULT_CSV:
        atomic_write_df(df, PERSIST_RESULT_CSV)
    summ = (
        df.groupby(["split","method","eval_type"], as_index=False)
          .accuracy.agg(["mean","std","count"]).reset_index()
    )
    atomic_write_df(summ, SUMMARY_CSV)
    if PERSIST_SUMMARY_CSV:
        atomic_write_df(summ, PERSIST_SUMMARY_CSV)
    print("Checkpointed:", split, seed, method, "| total rows:", len(df))


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## Run all splits × seeds

`robust_reference` is persisted because both DPO and confidence abstention depend on it.  
If you go to sleep, leave this cell running. Completed methods are checkpointed to Drive immediately.


In [5]:
split_col = {"ChemicalID":"ChemicalID", "GeneID":"GeneID", "DiseaseID":"DiseaseID"}

for split in SPLITS:
    for seed in SEEDS:
        print("\n" + "="*80)
        print("SPLIT", split, "SEED", seed)
        print("="*80)

        train_df, cal_df, test_df = make_split(paths, split_col[split], seed)
        cal_sets = make_eval_sets(cal_df, seed + 5000)
        eval_sets = make_eval_sets(test_df, seed)
        robust_adapter = ADAPTER_ROOT / f"robust_{split}_{seed}"

        # ------------------------------------------------------------
        # 0) Robust reference (needed by DPO + confidence baseline)
        # ------------------------------------------------------------
        need_robust_model = (
            not method_complete(rows, split, seed, "confidence_abstention")
            or not method_complete(rows, split, seed, "dpo")
            or not method_complete(rows, split, seed, "robust_reference")
        )

        robust_model = None
        if need_robust_model:
            if robust_adapter.exists() and any(robust_adapter.iterdir()):
                print("Loading persisted robust adapter:", robust_adapter)
                robust_model = load_adapter(robust_adapter, training=False)
            else:
                print("Training robust reference...")
                ds = make_robust_sft_dataset(train_df, seed)
                robust_model = train_sft(
                    ds, LOCAL_RESULT_DIR/f"{split}_{seed}_robust",
                    seed, save_adapter=robust_adapter
                )

        if not method_complete(rows, split, seed, "robust_reference"):
            print("Evaluating robust reference...")
            sc = evaluate_model(robust_model, eval_sets)
            upsert_method(rows, split, seed, "robust_reference", sc)

        # ------------------------------------------------------------
        # 1) Mixture SFT
        # ------------------------------------------------------------
        if not method_complete(rows, split, seed, "mixture_sft"):
            print("\nTraining mixture SFT...")
            mix_ds = make_mixture_sft_dataset(train_df, seed)
            mix_model = train_sft(
                mix_ds, LOCAL_RESULT_DIR/f"{split}_{seed}_mixture", seed
            )
            sc = evaluate_model(mix_model, eval_sets)
            upsert_method(
                rows, split, seed, "mixture_sft", sc,
                extra={"no_path_frac": MIX_NO_PATH_FRAC}
            )
            cleanup(mix_model)
        else:
            print("Skip completed: mixture_sft")

        # ------------------------------------------------------------
        # 2) Confidence-based abstention on robust SFT
        # ------------------------------------------------------------
        if not method_complete(rows, split, seed, "confidence_abstention"):
            print("\nCalibrating confidence abstention...")
            if robust_model is None:
                robust_model = load_adapter(robust_adapter, training=False)
            threshold = calibrate_confidence_threshold(robust_model, cal_sets)
            sc = evaluate_confidence_abstention(robust_model, eval_sets, threshold)
            upsert_method(
                rows, split, seed, "confidence_abstention", sc,
                extra={"confidence_threshold": threshold}
            )
        else:
            print("Skip completed: confidence_abstention")

        # Free inference model before DPO training.
        cleanup(robust_model)
        robust_model = None

        # ------------------------------------------------------------
        # 3) DPO from robust adapter
        # ------------------------------------------------------------
        if not method_complete(rows, split, seed, "dpo"):
            print("\nTraining DPO from robust reference...")
            dpo_ds = make_dpo_dataset(train_df, seed)
            dpo_model = train_dpo(
                dpo_ds, robust_adapter,
                LOCAL_RESULT_DIR/f"{split}_{seed}_dpo", seed
            )
            sc = evaluate_model(dpo_model, eval_sets)
            upsert_method(rows, split, seed, "dpo", sc)
            cleanup(dpo_model)
        else:
            print("Skip completed: dpo")

print("\nALL REQUESTED BLOCKS COMPLETE")
final_df = pd.DataFrame(rows)
display(final_df.tail(20))
display(pd.read_csv(SUMMARY_CSV))



SPLIT ChemicalID SEED 1
Training robust reference...


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,0.249234
40,0.017275
60,0.011857


clean 0.75
distractor_5 0.14
hard_no_path 1.0
lexical_no_path 1.0
counterfactual 0.69
Checkpointed: DiseaseID 3 dpo | total rows: 180

ALL REQUESTED BLOCKS COMPLETE


,split,seed,method,eval_type,accuracy,no_path_frac,confidence_threshold
160,DiseaseID,3,robust_reference,clean,0.82,NaN,NaN
161,DiseaseID,3,robust_reference,distractor_5,0.95,NaN,NaN
162,DiseaseID,3,robust_reference,hard_no_path,0.54,NaN,NaN
163,DiseaseID,3,robust_reference,lexical_no_path,0.39,NaN,NaN
164,DiseaseID,3,robust_reference,counterfactual,0.78,NaN,NaN
165,DiseaseID,3,mixture_sft,clean,0.57,0.25,NaN
166,DiseaseID,3,mixture_sft,distractor_5,0.87,0.25,NaN
167,DiseaseID,3,mixture_sft,hard_no_path,1.00,0.25,NaN
168,DiseaseID,3,mixture_sft,lexical_no_path,0.99,0.25,NaN
169,DiseaseID,3,mixture_sft,counterfactual,0.68,0.25,NaN


,index,split,method,eval_type,mean,std,count
0,0,ChemicalID,confidence_abstention,clean,0.190000,0.243311,3
1,1,ChemicalID,confidence_abstention,counterfactual,0.220000,0.252389,3
2,2,ChemicalID,confidence_abstention,distractor_5,0.803333,0.032146,3
3,3,ChemicalID,confidence_abstention,hard_no_path,0.906667,0.070946,3
4,4,ChemicalID,confidence_abstention,lexical_no_path,0.840000,0.111355,3
5,5,ChemicalID,dpo,clean,0.613333,0.518588,3
6,6,ChemicalID,dpo,counterfactual,0.633333,0.540031,3
7,7,ChemicalID,dpo,distractor_5,0.610000,0.524309,3
8,8,ChemicalID,dpo,hard_no_path,0.730000,0.467654,3
9,9,ChemicalID,dpo,lexical_no_path,0.726667,0.464794,3


## Quick paper-oriented comparison

The table below averages over splits and seeds. The key question is whether any intervention moves toward the upper-right corner: high Distractor-5 **and** high no-path accuracy.


In [ ]:
df = pd.read_csv(RESULT_CSV)
wide = (
    df.groupby(["method","eval_type"]).accuracy.mean()
      .unstack("eval_type")
      .reindex(columns=["clean","distractor_5","hard_no_path","lexical_no_path","counterfactual"])
)
wide["no_path_mean"] = wide[["hard_no_path","lexical_no_path"]].mean(axis=1)
wide["selection_sufficiency_hmean"] = (
    2 * wide["distractor_5"] * wide["no_path_mean"]
    / (wide["distractor_5"] + wide["no_path_mean"] + 1e-12)
)
display(wide.sort_values("selection_sufficiency_hmean", ascending=False))

print("Persistent result path:", PERSIST_RESULT_CSV)
print("Persistent summary path:", PERSIST_SUMMARY_CSV)
